In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from mdbmaster import MasterParams
from sys import prefix
mp = MasterParams(verbose=True)
io = FileIO()

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb


# DB-Specific

In [3]:
from mdblib import lastfm
mio   = lastfm.MusicDBIO(verbose=True)
apiio = lastfm.RawAPIData()
db    = mio.db

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb
MusicDBBaseDirs(db=LastFM)
   RawDataDir     = /Volumes/Piggy/Discog/artists-lastfm
   ModValDataDir  = /Volumes/Seagate/Discog/artists-lastfm-db
   MetaDataDir    = /Volumes/Seagate/Discog/artists-lastfm-db/metadata
   SummaryDataDir = /Users/tgadfort/Music/Discog/db-lastfm
ParseRawDataUtils(mdbdata, mdbdir) [LastFM]
LastFM ModValMetaData
  ==> Basic
  ==> Media
  ==> Genre
  ==> Link
  ==> Metric
  ==> Counts


# Perm Dir

In [4]:
def setupPermDir(db):
    mp = MasterParams(verbose=False)
    prefixDir = DirInfo(prefix)
    projDir   = prefixDir.join(mp.getProjectName())
    projDir.mkDir()
    libDir    = projDir.join("mdblib")
    libDir.mkDir()
    permDBDir = libDir.join(db)
    permDBDir.mkDir()
    return permDBDir
permDBDir = setupPermDir(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant LastFM DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/LastFM


# Master Files

In [5]:
from mdbbase import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
localArtistIDs     = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistIDs".format(db.lower()))
localArtistIDsData = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistIDsData".format(db.lower()))
localAlbums        = MusicDBData(path=permDir, fname="{0}SearchedForLocalAlbums".format(db.lower()))
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [6]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artist Search:       {0}".format(len(localArtists.get())))
print("   Local Artist IDs:          {0}".format(len(localArtistIDs.get())))
print("   Local Artist IDs Data:     {0}".format(len(localArtistIDsData.get())))
print("   Local Album Search:        {0}".format(len(localAlbums.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(mio.data.getArtistIDToNameData())))

LastFM Search Results
   Global Master Search:      48674
   Global Master Search Data: 3200
   Local Artist Search:       134223
   Local Artist IDs:          0
   Local Artist IDs Data:     0
   Local Album Search:        187656
   Errors:                    0
   Known Summary IDs:         190384


# Download Artist Search Data

In [7]:
mio   = lastfm.MusicDBIO(verbose=False)
apiio = lastfm.RawAPIData(debug=True)

LastFM API(Key=0017f8ea36758d0923766b8121a98984)


## Find Artists To Download

In [24]:
from musicdb import MusicDBIO
from pandas import Series
mdbio = MusicDBIO()
mmeDF = mdbio.getData()
unmatchedArtistNames = Series(mmeDF["ArtistName"].unique())
searchedForMasterArtists = masterArtists.get()
artistNamesToGet = unmatchedArtistNames[~unmatchedArtistNames.isin(searchedForMasterArtists.keys())]

print("{0} Search Results".format(db))
print("   Known Master Artist Names:    {0}".format(len(unmatchedArtistNames)))
print("   Known Spotify Artist Names:   {0}".format(len(searchedForMasterArtists)))
print("   Artist Names To Get:          {0}".format(len(artistNamesToGet)))

LastFM Search Results
   Known Master Artist Names:    754727
   Known Spotify Artist Names:   95557
   Artist Names To Get:          659170


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "2:30pm")
#tt = TermTime("today", "9:15pm")
n  = 0

searchedForMasterArtistsData = masterArtistsData.get()
searchedForMasterArtists     = masterArtists.get()
searchedForErrors            = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if searchedForMasterArtists.get(artistName) is not None:
        continue

    response = apiio.getArtistSearchResults(artistName=artistName)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((idx,artistName)))
        searchedForMasterArtists[artistName] = True
        apiio.sleep(3.5)
        continue
        
    searchedForMasterArtistsData[artistName] = response
    searchedForMasterArtists[artistName] = True
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForMasterArtists), db))
        masterArtists.save(data=searchedForMasterArtists)
        masterArtistsData.save(data=searchedForMasterArtistsData)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving [{0} / {1}] {2} Searched For Artist IDs".format(len(searchedForMasterArtists), len(searchedForMasterArtistsData), db))
masterArtists.save(data=searchedForMasterArtists)
masterArtistsData.save(data=searchedForMasterArtistsData)

Starting Process [Getting LastFM ArtistIDs]    ==> Time Is 2022-03-07 11:09:29
========================= termTime(day=tomorrow,time=11:00am) =========================
   ====> Terminate Time Set To 2022-03-08 11:00:00 <====
   ====> Will Terminate Process 23 Hours and 50 Minutes From Now
Searching For Jeff Green                                        486     30
Searching For Adam Kowalewski                                   59      30
Searching For Zoff                                              330     30
Searching For Åke Söderblom                                     40      30
Searching For 1400 Points de Suture                             4       4
Searching For Sergio Calzoni                                    2       2
Searching For xKore                                             506     30
Searching For Gustavo Adolfo Bécquer                            2       2
Searching For Reboelje                                          5       5
Searching For Frans Nieuwenhuis         

Searching For Alain Bouche                                      12      12
Searching For Cino Ghedin                                       6       6
Searching For Djazia Satour                                     15      15
Searching For Ermavilo                                          16      16
Searching For Lollies                                           160     30
Searching For Bill Wren                                         17      17
Searching For Hans Feigenwinter                                 25      25
Searching For Fred Mühlböck                                   0       0
Error ==> (95649, 'Fred Mühlböck')
Searching For J. C. Grimshaw                                    5       5
Searching For Louise Vant                                       5       5
Searching For Bob Vrieling                                      1       1
Searching For Jeros                                             108     30
Searching For Reduxtion                                         3   

Searching For Perle Lama                                        277     30
Searching For Thomas Ragsdale                                   22      22
Searching For Juraj Fortwangler                                 1       1
Searching For Stanley Lippitt                                   1       1
Searching For Anamari                                           527     30
Searching For Seno                                              6985    30
Searching For Hanuš Bartoň                                    0       0
Error ==> (95736, 'Hanuš Bartoň')
Searching For Winston Morris                                    74      30
Searching For Jonathan Michell                                  141     30
Searching For Wolfgang Riedelbauch                              31      30
Searching For Geir Åge Johnsen                                 0       0
Error ==> (95740, 'Geir Åge Johnsen')
Searching For Fred Holstein                                     16      16
Searching For Christian Skjød

Searching For Laza Ristovski                                    29      29
Searching For Mr. Bloe                                          58      30
Searching For Lena Agree                                        2       2
Searching For Kung                                              8695    30
Searching For Māris Sirmais                                    1       1
Searching For Alexis Descharmes                                 55      30
Searching For Jan Hellriegel                                    5       5
Searching For Anthony Robson                                    647     30
Searching For Ernie Rettino                                     25      25
Searching For David Mansfield                                   119     30
Searching For Nadir Khayat                                      113     30
Searching For Trespassers W                                     168     30
Searching For Joe Moody                                         76      30
Searching For Fred Mitchell 

Searching For JeanJass                                          405     30
Searching For Fares Karam                                       133     30
Searching For Henri Christiné                                  0       0
Error ==> (95908, 'Henri Christiné')
Searching For Ryan Carniaux                                     46      30
Searching For HammAli & Navai                                   608     30
Searching For Orchestra da Camera di Genova                     18      18
Searching For Ari Pulkkinen                                     94      30
Searching For Young Doe                                         637     30
Searching For David Monsoh                                      0       0
Error ==> (95914, 'David Monsoh')
Searching For Mike FitzHenry                                    0       0
Error ==> (95915, 'Mike FitzHenry')
Searching For Mike Majewski                                     3       3
Searching For Tiara Thomas                                      264    

...

Searching For The Gents                                         375     30
9500/?     : Process [Getting LastFM ArtistIDs] Has Run For 9 Hours and 12 Minutes.
Saving 106304 LastFM Searched For Artist IDs
Searching For Kenneth Wannberg                                  7       7
Searching For Savaria Symphony Orchestra                        55      30
Searching For John Ward                                         820     30
Searching For Driss El Maloumi                                  367     30
Searching For Jesse Litwa                                       0       0
Error ==> (106308, 'Jesse Litwa')
Searching For Thebe Kgositsile                                  24      24
Searching For Theo Hakola                                       27      27
Searching For Beatriz Klien                                     26      26
Searching For Franka Lampe                                      6       6
Searching For Abi Ofarim                                        131     30
Searching For D

Searching For Osjan                                             42      30
Searching For Stabscotch                                        2       2
Searching For Boys Age                                          136     30
Searching For Wolfgang Jäger                                    6       6
Searching For Jeff Knowler                                      25      25
Searching For Frédéric Lansac                                 0       0
Error ==> (106394, 'Frédéric Lansac')
Searching For Stéphane Caillat                                  55      30
Searching For Richard Maranda                                   2       2
Searching For Andreas Ralsgård                                 0       0
Error ==> (106397, 'Andreas Ralsgård')
Searching For Mad Mark                                          2980    30
Searching For Vasilis Petridis                                  2       2
Searching For Al Schultz                                        14      14
Searching For José Marcel

Searching For Ornette                                           1679    30
Searching For Pictures of Vernon                                6       6
Searching For Ronnie Baker Brooks                               135     30
Searching For William Stewart                                   311     30
Searching For John Wesley Work                                  12      12
Searching For Robert Julian Horky                               4       4
Searching For Chase Buch                                        85      30
Searching For Arnaud Spitz                                      0       0
Error ==> (106482, 'Arnaud Spitz')
Searching For Bjorn Englen                                      4       4
Searching For Florian Schanze                                   0       0
Error ==> (106484, 'Florian Schanze')
Searching For Gerry DeVeaux                                     91      30
Searching For Anthony Bernard                                   257     30
Searching For Perry Hughes      

Searching For Joe "Red" Clark                                   5       5
Searching For GO Twice                                          31      30
Searching For Seth Baer                                         13      13
Searching For Makám                                            6       6
Searching For David Fontaine                                    13      13
Searching For Großer Chor des Berliner Rundfunks                51      30
Searching For Brian Egeness                                     3       3
Searching For Will Joss                                         15      15
Searching For Brenda Lee Eager                                  66      30
Searching For Franz Litschauer                                  29      29
Searching For Viktor Simcisko                                   93      30
Searching For Grampall Jookabox                                 17      17
Searching For Corey Glover                                      256     30
Searching For Nathalie      

Searching For Horologium                                        50      30
Searching For EeL                                               10116   30
Searching For Katerine Avgoustakis                              32      30
Searching For Quadrajets                                        9       9
Searching For Gabe Baltazar                                     11      11
Searching For Admiral Angry                                     7       7
Searching For Eleventh He Reaches London                        7       7
Searching For Paktofonika                                       1200    30
Searching For Gotthold Ephraim Lessing                          16      16
Searching For Tomáš Král                                     0       0
Error ==> (106660, 'Tomáš Král')
Searching For Nestore Catalani                                  46      30
Searching For Arthur Motta                                      16      16
Searching For Shop Assistants                                   53  

In [21]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        df = DataFrame(getFlatList([v for k,v in mad.items()]))
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [22]:
mad = masterArtistsData.get()
df  = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = lastfm.MusicDBIO(verbose=False).data.getSearchArtistData()    
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['listeners'] = searchDF['listeners'].astype(int)    
    searchDF = searchDF.sort_values(by='listeners', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    lastfm.MusicDBIO(verbose=False).data.saveSearchArtistData(data=searchDF)
else:
    print("No new artists")

Found 490054 New Artists
Found 1160364 Previous Artists
Found 1650418 Total Artists
Found 1600715 Unique Artists
Found 1600715 Unique + Sorted Artists
Saving Data


In [23]:
masterArtistsData.save(data={})

## Download Data By Artist IDs

In [ ]:
print("{0} Search Results".format(db))
print("   Known Summary IDs:    {0}".format(len(knownArtists)))
artistIDsToGet = knownArtists[knownArtists.str.len() < 1].index
print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))
artistIDsToGet = artistIDsToGet[~artistIDsToGet.isin(searchedForArtistIDs.keys())]
print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "9:15pm")
n  = 0
searchedForArtistIDsData = getSearchedForLocalArtistIDsData()
searchedForArtistIDs = getSearchedForLocalArtistIDs()
searchedForErrors = getSearchedForErrors()
for i,dbID in enumerate(artistIDsToGet):
    if searchedForArtistIDs.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    response = apiio.getArtistIDLookupResults(artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = "?"
        saveSearchedForErrors(searchedForErrors)
        apiio.sleep(5)
        continue
        
    searchedForArtistIDsData[dbID] = response
    searchedForArtistIDs[dbID] = True
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForArtistIDs), db))
        saveSearchedForLocalArtistIDs(searchedForArtistIDs)
        saveSearchedForLocalArtistIDsData(searchedForArtistIDsData)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForArtistIDs), db))
saveSearchedForLocalArtistIDs(searchedForArtistIDs)
saveSearchedForLocalArtistIDsData(searchedForArtistIDsData)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

In [ ]:
from pandas import DataFrame, Series, concat

def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

def getArtistIDsDataFrame(aids):
    df = None
    if isinstance(aids,dict):
        df = DataFrame([v for k,v in aids.items()])[0].apply(Series) if len(aids) > 0 else None
    return df
        
def getResponseDataFrame(aids):
    df = getArtistIDsDataFrame(aids)
    if not isinstance(df,DataFrame):
        return None
    #df = DataFrame(response)
    df["followers"] = df["followers"].apply(getFollowers)
    df.index = df['sid']
    df.drop(['sid'], axis=1, inplace=True)
    return df

In [ ]:
aids = getSearchedForLocalArtistIDsData()
df   = getResponseDataFrame(aids)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getSearchArtistData()
    print("Found {0} Previous Artists".format(searchDF.shape[0]))
    searchDF = concat([searchDF,df])
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF = searchDF.sort_values(by='popularity', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveSearchArtistData(data=searchDF)
else:
    print("No new artists")

In [ ]:
saveSearchedForLocalArtistIDsData({})

# Move Local

In [ ]:
from mdblib.spotify import MusicDBIO
mio = MusicDBIO()

In [ ]:
from mdblib.spotify import moveLocalFiles

In [ ]:
moveLocalFiles()

# Download Album Data

## Find Albums To Get

In [ ]:
print("{0} Search Results".format(db))
print("   Known Summary IDs:    {0}".format(len(knownArtists)))
artistNamesToGet = knownArtists[~knownArtists.index.isin(searchedForAlbums.keys())]
artistNamesToGet = artistNamesToGet #.sample(frac=1)
print("   Artists IDs To Get:   {0}".format(len(artistNamesToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Albums".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "4:00pm")
n  = 0
searchedForAlbums = getSearchedForLocalAlbums()
searchedForErrors = getSearchedForErrors()
for i,(dbID,artistName) in enumerate(artistNamesToGet.iteritems()):    
    if searchedForAlbums.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    modVal=mio.mv.get(dbID)
    if mio.data.getRawArtistAlbumFilename(modVal, dbID).exists():
        searchedForAlbums[dbID] = artistName
        continue
    
    response = apiio.getArtistAlbums(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = artistName
        saveSearchedForErrors(searchedForErrors)
        apiio.sleep(5)
        continue
        
    mio.data.saveRawArtistAlbumData(data=response, modval=modVal, dbID=dbID)        
    searchedForAlbums[dbID] = artistName
    apiio.sleep(7.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
        saveSearchedForLocalAlbums(searchedForAlbums)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
    if n % 25 == 0:
        apiio.sleep(30.0)
    
ts.stop()
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
saveSearchedForLocalAlbums(searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

In [ ]:
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
saveSearchedForLocalAlbums(searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

# Move Local Files

In [ ]:
from mdblib import spotify
spotify.moveLocalFiles()

# Parse What We Got

In [ ]:
%autoreload
from mdbutils import poolIO
mio = discogs.MusicDBIO(debug=False)
poolIO(parseFunction=mio.prd.parseAlbumData, modVals=range(100))

# Download Artist Info Data

## Determine Artists To Download

## Download Artist Info

# Download Artist Albums Data

## Determine Artists To Download

In [ ]:
spotifyArtists = io.get("/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
spotifyArtists = spotifyArtists.sort_values(by='popularity', ascending=False)

In [ ]:
searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = spotifyArtists[~spotifyArtists.index.isin(searchedForArtists.index)]['name']
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
from pandas import Series
meu = masterManualEntriesUtils()

knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    sids = [sid for sid in sids if len(sid) == 22]
    spotifyToGet.update({sid: name for sid in sids})
knownSpotify = Series(spotifyToGet)
print("Found {0} Known Spotify Artists".format(len(knownSpotify)))

searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = knownSpotify[~knownSpotify.index.isin(searchedForArtists.index)]
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
from timeUtils import timestat, termTime

dbIO = dbg.getDBIO("Spotify")
api  = apig.getDBAPIData("Spotify")

searchedForArtists = io.get("spotifySearchedForArtists.p")
errs = io.get("spotifyErrorsSearchedForArtists.p")
ts = timestat("Downloading Spotify Data")
#tt = termTime("today", "10:00pm")
tt = termTime("tomorrow", "10:00pm")
#tt = termTime("tomorrow", "7:00am")
#tt = termTime("today", "9:30am")
n  = 0
for i,(dbID,artistName) in enumerate(artistNames.iteritems()):    
    if searchedForArtists.get(dbID) is not None:
        continue
    if errs.get(dbID) is not None:
        continue
    if not dbIO.isArtistAlbumsKnown(dbID) or True:
        response = api.getArtistAlbums(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        errs[dbID] = artistName
        io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")
        api.sleep(5)
        continue
    dbIO.saveArtistAlbums(dbID, response)
    searchedForArtists[dbID] = artistName
    api.sleep(7.5)
    n += 1
        
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        print("="*150)
        api.sleep(7.5)
        if tt.isFinished():
            break
    
    if n % 25 == 0:
        api.sleep(30.0)
    
ts.stop()
print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} Spotify Errors For Artists".format(len(errs)))
    io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")

# Extra

In [ ]:
  
    api.sleep(7.5)
    n += 1
    
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n,N=stop)
        print("="*150)
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        api.sleep(7.5)
    
    if (i+1) % 25 == 0:
        sleep(30.0)

In [ ]:
######################################################
# Juypter
######################################################
%load_ext autoreload
%autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("""<style>div.output_area{max-height:10000px;overflow:scroll;}</style>"""))

###########################################################################
## Warnings
###########################################################################
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=FutureWarning) 


###########################################################################
## Utils
###########################################################################
from ioUtils import getFile, saveFile
from webUtils import getHTML
from searchUtils import findExt
from fsUtils import setFile, isFile, fileUtil
from fileUtils import getBaseFilename
from timeUtils import timestat
from fileIO import fileIO
from dbUtils import utilsSpotifyCharts


###########################################################################
## General
###########################################################################
import urllib
from glob import glob
from time import sleep
from io import StringIO
from pandas import Series,DataFrame,concat,read_csv,date_range
from datetime import datetime
from urllib.parse import unquote, urlparse
from pathlib import PurePosixPath



######################################################
# Versions
######################################################
import sys
print("Python: {0}".format(sys.version))
import datetime as dt
start = dt.datetime.now()

print("Notebook Last Run Initiated: "+str(start))

# Global

In [ ]:
io = fileIO()
from masterDBGate import masterDBGate
mdbGate = masterDBGate()

from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()

from spotipy.oauth2 import SpotifyClientCredentials
import spotipy
auth_manager=SpotifyClientCredentials(client_id="61e441c3b90c4873aa0e6b9582564f95", client_secret="ae0d0f968bf443fdac1d9ac6ef65fc0f")
sp = spotipy.Spotify(auth_manager=auth_manager)

# Spotify API

In [ ]:
# !pip install spotipy

In [ ]:
from artistSpotify import artistSpotify
#from artistIDBase import artistIDSpotify
from dbBase import dbBase

from fsUtils import dirUtil,fileUtil
from artistIDBase import dbArtistID
from artistModValue import artistModValue
from masterArtistNameCorrection import masterArtistNameCorrection


##################################################################################################################
# Base Class
##################################################################################################################
class dbSpotify:
    def __init__(self):
        self.db     = "Spotify"
        self.disc   = dbBase(self.db.lower())
        self.artist = artistSpotify(self.disc)
        self.aid    = dbArtistID(self.db)

        self.getpsid   = self.aid.getpsid
        self.getModVal = self.aid.getModVal
        
        self.io = fileIO()
        self.manc = masterArtistNameCorrection()
        
        
    def getSearchDir(self, artistName):
        psID = self.getpsid(self.manc.clean(artistName))
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(psID)))
        searchDir = dirUtil(modValDir).join("search")
        return searchDir
        
    def getArtistSearchSavename(self, artistName):
        psID = self.getpsid(self.manc.clean(artistName))
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(psID)))
        searchDir = dirUtil(modValDir).join("search")
        return fileUtil(searchDir).join("{0}.p".format(psID))
    
    def saveArtistSearch(self, artistName, artistSearch):
        self.io.save(idata=artistSearch, ifile=self.getArtistSearchSavename(artistName))
        
        
    def getArtistAlbumsSavename(self, artistID):
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(artistID)))
        return fileUtil(modValDir).join("{0}.p".format(artistID))
    
    def saveArtistAlbums(self, artistID, artistAlbums):
        self.io.save(idata=artistAlbums, ifile=self.getArtistAlbumsSavename(artistID))

In [ ]:
import requests
from time import sleep

class apiIO:
    def __init__(self, name):
        self.name = name
        self.code = None
        self.url = None
        self.response = None
        
    def get(self, url):
        self.url       = url
        try:
            self.response  = requests.get(url)
        except:
            self.response  = None
            self.code      = 0
            return {}
            
        self.code      = self.response.status_code
        
        try:
            json_data      = self.response.json() if self.response.status_code == 200 else {}
        except:
            json_data      = {}
        return json_data
    
    def getResponse(self):
        return self.response
    
    def getURL(self):
        return self.url
    
    def getStatus(self):
        return self.code
    
    def sleep(self, value):
        sleep(value)

In [ ]:
def getArtistTrackResults(artistName, artistID, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    searchResults  = []
    offset         = 0
    sleep(0.5)
    requestResult  = sp.artist_top_tracks(artistID, limit=limit, offset=offset)
    offset += limit
    
    if requestResult is None:
        return None
    totalResults   = requestResult.get('total', -1)
    nextURL        = requestResult.get('next', None)
    try:
        searchResults += requestResult['items']
    except:
        return None
    print("   ===> {0: <8}   {1}".format("[{0}]".format(totalResults), len(searchResults)), end=" ")
    while nextURL is not None:
        requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
        offset += limit
        try:
            searchResults += requestResult['items']
        except:
            return None
        nextURL        = requestResult.get('next', None)
        if nextURL:
            #print("{0}".format(len(searchResults)), end="")
            print(".", end="")
            sleep(1.0)
    print(" {0}".format(len(searchResults)))
    return searchResults

In [ ]:
def getArtistAlbumResults(artistName, artistID, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    searchResults  = []
    offset         = 0
    sleep(0.5)
    requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
    offset += limit
    
    if requestResult is None:
        return None
    totalResults   = requestResult.get('total', -1)
    href           = requestResult.get('href')
    artistID       = PurePosixPath(unquote(urlparse(href).path)).parts[3]
    nextURL        = requestResult.get('next', None)
    try:
        searchResults += requestResult['items']
    except:
        return None
    print("   ===> {0: <8}   {1}".format("[{0}]".format(totalResults), len(searchResults)), end=" ")
    while nextURL is not None:
        sleep(3.5)
        requestResult  = sp.artist_albums(artistID, limit=limit, offset=offset)
        offset += limit
        try:
            searchResults += requestResult['items']
        except:
            return None
        nextURL        = requestResult.get('next', None)
        if nextURL:
            #print("{0}".format(len(searchResults)), end="")
            print(".", end="")
    print(" {0}".format(len(searchResults)))
    
    albums = [spotifyAlbumRecord(album).get() for album in searchResults]
    retval = {'href': href, 'artistID': artistID, 'albums': albums}
    return retval

In [ ]:
def getArtistSearchResults(artistName, limit=50):
    print("Searching For {0: <30}".format(artistName), end="")
    sleep(0.25)
    result = sp.search(artistName, limit=limit, type='artist')
    
    artists = result.get('artists', {}) if isinstance(result,dict) else {}
    href    = artists.get('href')
    total   = artists.get('total')
    nextURL = artists.get('next')
    prevURL = artists.get('previous')
    items   = artists.get('items', [])
    retval  = [spotifyArtistRecord(item).get() for item in items]
    print(len(retval))
    
    return retval

# Artist Search

In [ ]:
from glob import glob
from masterUtils import masterUtils
mu = masterUtils()
disc = mu.getDisc("Spotify")
ts = timestat("Finding Search Files")
files = glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")
ts.stop()

In [ ]:
from fileIO import fileIO
io = fileIO()
#io.save(idata=files, ifile="spotifyFiles.p")
files = io.get("spotifyFiles.p")
len(files)

In [ ]:
from fsUtils import fileUtil, mkDir, dirUtil, moveFile
dbIO = dbSpotify()
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    artistName = fileUtil(ifile).basename
    searchDir = dbIO.getSearchDir(artistName)
    if not searchDir.exists():
        print("Making {0}".format(searchDir))
        mkDir(searchDir)
    savename = dbIO.getArtistSearchSavename(artistName=artistName)
    moveFile(ifile,savename)
    
    if (i+1) % 10000 == 0 or (i+1) == 1000 or (i+1) == 100:
        ts.update(n=i+1, N=len(files))
ts.stop()

In [ ]:
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
ts = timestat("Getting Searched For Artists")
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
ts.stop()
print("Found {0} Searched For Artists".format(len(searchedForArtists)))

In [ ]:
from fsUtils import fileUtil
from time import sleep
from masterArtistNameCorrection import masterArtistNameCorrection
manc = masterArtistNameCorrection()
ts = timestat("Getting Artists To Download")
mmeDF = meu.getDF()
artistNames = mmeDF["ArtistName"].apply(manc.clean)
print("Found {0} Artists".format(artistNames.shape[0]))
artistNamesToDownload = artistNames[~artistNames.isin(searchedForArtists)]
print("Found {0} Artists To Download".format(artistNamesToDownload.shape[0]))
ts.stop()

In [ ]:
toget = {}
for i,(idx,artistName) in enumerate(artistNamesToDownload.iteritems()):
    if i >= 2500:
        break
    savefile = fileUtil("spotify/artists").join("{0}.p".format(manc.clean(artistName)))
    if savefile.exists():
        continue
    toget[savefile] = artistName
print("Need To Download {0} Artists".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,artistName) in enumerate(toget.items()):
    if savefile.exists():
        continue
    if nErr >= 5:
        break
    result = getArtistSearchResults(artistName)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60*nErr)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(2.5)
    
    if (i+1) % 25 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(1.5)
        print("")
    
    if (i+1) % 100 == 0:
        sleep(120)
ts.stop()

## Artist Results

In [ ]:
from artistModValue import artistModValue
amv = artistModValue()

# Move Artist Files

In [ ]:
from fsUtils import dirUtil,fileUtil,moveFile

files = glob("spotify/artists/*.p")
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    if (i+1) % 1000 == 0:
        ts.update(n=i+1,N=len(files))
        
    src = fileUtil(ifile)
    dst = dirUtil("/Volumes/Piggy/Discog/artists-spotify/artists").join(src.name)
    if dst.exists():
        continue
    moveFile(src.path, dst)

In [ ]:
searchedForArtists = Series([fileUtil(x).basename for x in glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")])
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
print("Found {0} Searched For Artists".format(len(searchedForArtists)))

# Move Album Files

In [ ]:
from fsUtils import dirUtil,fileUtil,moveFile,mkDir
from artistModValue import artistModValue
amv = artistModValue()

files = glob("spotify/albums/*.p")
ts = timestat("Moving {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    if (i+1) % 1000 == 0:
        ts.update(n=i+1,N=len(files))
        
    src = fileUtil(ifile)
    albumsData = io.get(ifile)
    artistID = albumsData['artistID']
    modValue = amv.getModVal(artistID)
    modDir = dirUtil("/Volumes/Piggy/Discog/artists-spotify/").join(str(modValue))
    if not modDir.exists():
        mkDir(modDir)
    dst = dirUtil(modDir).join("{0}.p".format(artistID))
    if dst.exists():
        continue
    print("  ==>  {0}".format(dst))
    moveFile(src.path, dst)

# Create Master Artist ID List

In [ ]:
from glob import glob
files = glob("/Volumes/Piggy/Discog/artists-spotify/artists/*.p")
artistsData = []
N = len(files)
ts = timestat("Loading {0} Spotify Artist Search Files".format(N))
for i,ifile in enumerate(files):    
    artistsData += io.get(ifile)
    if (i+1) % 5000 == 0 or (i+1) == 1000:
        ts.update(n=i+1, N=N)
ts.stop()

In [ ]:
def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

df = DataFrame(artistsData)
df["followers"] = df["followers"].apply(getFollowers)
df.index = df['sid']
df.drop(['sid'], axis=1, inplace=True)
df = df[~df.index.duplicated(keep='first')]
df = df.sort_values(by='popularity', ascending=False)

print("Saving {0} Spotify Artists Data".format(df.shape[0]))
io.save(idata=df, ifile="/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
df.head(10)

In [ ]:
print(df.shape)

# Artist Albums

In [ ]:
spotifyArtists = io.get("/Volumes/Piggy/Discog/artists-spotify/search/spotifyArtistsData.p")
spotifyArtists = spotifyArtists.sort_values(by='popularity', ascending=False)

## Download Albums Data

In [ ]:
spotifyArtists['popularity'].hist(bins=100, log='y')

In [ ]:
spotifyArtists[spotifyArtists["popularity"] >= 60].shape

In [ ]:
knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    for sid in sids:
        if len(sid) == 22:
            spotifyToGet[sid] = name
spotifyToGet = Series(spotifyToGet)
#spotifyToGet = knownSpotify["ArtistName"]
#spotifyToGet.index = list(knownSpotify["Spotify"].values)
#spotifyToGet.name = "name"
#spotifyToGet

In [ ]:
toget = {}
from artistModValue import artistModValue
from masterUtils import masterUtils
from fsUtils import dirUtil, mkDir
mu   = masterUtils()
amv  = artistModValue()
disc = mu.discs["Spotify"]


#spotifyToGet = spotifyArtists[spotifyArtists["popularity"] >= 60]
#print("Found {0} Spotify Artists".format(spotifyToGet.shape[0]))
#for i,(artistID,artistName) in enumerate(spotifyToGet["name"].iteritems()):

print("Found {0} Spotify Artists".format(spotifyToGet.shape[0]))
nKnown = 0
for i,(artistID,artistName) in enumerate(spotifyToGet.iteritems()):
    modVal   = amv.getModVal(artistID)
    modDir   = dirUtil(disc.getArtistsDir()).join(str(modVal))
    if not modDir.exists():
        print("Making {0}".format(modDir))
        mkDir(modDir)
    savefile = dirUtil(modDir).join("{0}.p".format(artistID))
    if savefile.exists():
        nKnown += 1
        continue
    #print("{0: <25}{1: <20} ({2})".format(artistName,artistID,modVal))
    toget[savefile] = (artistName,artistID)
    if len(toget) >= 2500:
        break
print("Found {0} Known Artists".format(nKnown))
print("Need To Download {0} Artist Albums".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,(artistName,artistID)) in enumerate(toget.items()):
    if nErr >= 2:
        break
    result = getArtistAlbumResults(artistName=artistName, artistID=artistID)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(7.5)
    
    if (i+1) % 5 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(15.0)
        print("")
    
    if (i+1) % 25 == 0:
        sleep(30.0)
ts.stop()

## Determine Artists To Download

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()

knownSpotify = meu.getDF()[["Spotify", "ArtistName"]].copy(deep=True)
knownSpotify = knownSpotify[knownSpotify["Spotify"].notna()]
spotifyToGet = {}
for idx,row in knownSpotify.iterrows():
    sid  = row["Spotify"]
    name = row["ArtistName"]
    sids = sid if isinstance(sid,list) else [sid]
    for sid in sids:
        if len(sid) == 22:
            spotifyToGet[sid] = name
knownSpotify = Series(spotifyToGet)
print("Found {0} Known Spotify Artists".format(len(knownSpotify)))

searchedForArtists = Series(io.get("spotifySearchedForArtists.p"))
print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

artistNames = knownSpotify[~knownSpotify.index.isin(searchedForArtists.index)]
print("Found {0} Spotify Artists To Download".format(len(artistNames)))

In [ ]:
io.save(idata={}, ifile="spotifyErrorsSearchedForArtists.p")

In [ ]:
stop = 150
dbIO = dbSpotify()
api  = spotifyAPIIO()
ts   = timestat("Getting Artist Data From Spotify API")
n    = 0

searchedForArtists = io.get("spotifySearchedForArtists.p")
errs = io.get("spotifyErrorsSearchedForArtists.p")
for i,(artistID,artistName) in enumerate(artistNames.iteritems()):
    if searchedForArtists.get(artistID) is not None:
        continue
    if errs.get(artistID) is not None:
        continue
    savename = dbIO.getArtistAlbumsSavename(artistID)
    if savename.exists():
        searchedForArtists[artistID] = artistName 
        continue
        
    #print(artistID,artistName,savename)
    response = api.getArtistAlbums(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        errs[artistID] = artistName
        io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")
        sleep(5)
        continue
    dbIO.saveArtistAlbums(artistID=artistID, artistAlbums=response)
    searchedForArtists[artistID] = artistName    
    api.sleep(7.5)
    n += 1
    
    if n % 5 == 0:
        api.sleep(7.5)
        print("="*150)
        ts.update(n=n,N=stop)
        print("="*150)
        io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
        api.sleep(7.5)
    
    if (i+1) % 25 == 0:
        sleep(30.0)
    
ts.stop()
print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} Spotify Error Artists".format(len(errs)))
    io.save(idata=errs, ifile="spotifyErrorsSearchedForArtists.p")

In [ ]:
if False:
    searchedForArtists = io.get("spotifySearchedForArtists.p")
    print("Found {0} Spotify Searched For Artists".format(len(searchedForArtists)))

    ts = timestat("Finding Spotify Saves")
    first = True
    for i,(artistID,artistName) in enumerate(spotifyToGet.iteritems()):
        if searchedForArtists.get(artistID) is not None:
            continue
        if first:
            print("Start: {0}".format(i))
            first = False
        if dbIO.getArtistAlbumsSavename(artistID).exists():
            searchedForArtists[artistID] = artistName
            if len(searchedForArtists) % 5000 == 0:
                break
        else:
            break

        if (i+1) % 100 == 0:
            ts.update(n=i+1,N=len(spotifyToGet))        
            #io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")

    ts.stop()
    print("Saving {0} Spotify Searched For Artists".format(len(searchedForArtists)))
    io.save(idata=searchedForArtists, ifile="spotifySearchedForArtists.p")

In [ ]:
spotifyArtists.head()

In [ ]:
from dbArtistsBase import dbArtistsBase
from fileUtils import getBaseFilename
from fsUtils import isFile, setFile, setDir
from ioUtils import getFile, saveFile
from timeUtils import timestat
from sys import prefix
import urllib
from time import sleep
from webUtils import getHTML
from pandas import Series
    
from fileIO import fileIO
from fsUtils import fileUtil

In [ ]:
from dbArtistsParse import dbArtistsSpotifyAPI
from dbArtistsSpotify import dbArtistsSpotify
dbAP = dbArtistsSpotifyAPI(dbArtistsSpotify())

In [ ]:
for modVal in range(100):
    dbAP.parse(modVal=modVal, expr='< 0 Days', force=False)
    
#for modVal in range(100):
#    dbAP.parseSearch(modVal, expr='< 0 Days', force=False)

In [ ]:
from artistModValue import artistModValue
amv = artistModValue()
modVal = 91
idx = artistSearchData.reset_index()['sid'].apply(amv.getModVal) == modVal
idx.index = artistSearchData.index
artistSearchData[idx]

In [ ]:
artistSearchData.head()

In [ ]:
art = artistSpotify()
art.getData(inputData).show()

In [ ]:
io.get("/Users/tgadfort/Music/Discog/artists-spotify-db/91-DB.p").get('1uNFoZAHBGtllmzznpCI3s').show()

In [ ]:
from fsUtils import fileUtil
from time import sleep
from masterArtistNameCorrection import masterArtistNameCorrection
manc = masterArtistNameCorrection()
mmeDF = meu.getDF()

In [ ]:

toget = {}
for i,(artistID,artistName) in enumerate(spotifyArtists["name"].iteritems()):
    if i >= 5:
        break
    savefile = fileUtil("spotify/albums").join("{0}--{1}.p".format(artistID,manc.clean(artistName)))
    if savefile.exists():
        pass
        #continue
    toget[savefile] = (artistName,artistID)
print("Need To Download {0} Artist Albums".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Spotify Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,(artistName,artistID)) in enumerate(toget.items()):
    if nErr >= 2:
        break
    result = getArtistAlbumResults(artistName=artistName, artistID=artistID)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(3.0)
    
    if (i+1) % 5 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(1.5)
        print("")
ts.stop()

## Artist Albums Results

In [ ]:
from glob import glob
files = glob("spotify/albums/*.p")
artistAlbumsData = {}
for ifile in files:
    albumsData   = io.get(ifile)
    artistID     = albumsData['artistID']
    artistAlbums = albumsData['albums']
    if artistAlbumsData.get(artistID) is not None:
        raise ValueError("Already have data for {0}".format(artistID))
    artistAlbumsData[artistID] = artistAlbums

In [ ]:
retval = getArtistAlbumResults(artistID='46sPgFJpEcoxUJNr1ouR9G', artistName='dummy')

In [ ]:
s = Series(artistAlbumsData)

In [ ]:
s.apply(len)

In [ ]:
retval = sp.artist_albums('13y7CgLHjMVRMDqxdx0Xdo', limit=10, offset=0)

In [ ]:
retval['href']

In [ ]:
href='https://api.spotify.com/v1/artists/13y7CgLHjMVRMDqxdx0Xdo/albums?offset=0&limit=10&include_groups=album,single,compilation,appears_on'
href

In [ ]:
from urllib.parse import unquote, urlparse
from pathlib import PurePosixPath

url = 'http://www.example.com/hithere/something/else'

PurePosixPath(unquote(urlparse(href).path)).parts[3]

In [ ]:
from urlparse import urlparse, parse_qs
s = "https://xx.com/question/index?qid=2ss2830AA38Wng"
parse_qs(urlparse(s).query)['qid'][0]

In [ ]:
rParen = r'\((.*?)\)+'
rFeat  = r'\b(feat.\s|Feat.\s|with\s)\b'
rSuffix= r'\s-\sRemix'

In [ ]:
featureValue = regex.search(rFeat, parenValue.group())


In [ ]:
retval = sp.artist_top_tracks('60d24wfXkVzDSfLS6hyCjZ')

In [ ]:
help(sp.tracks)

In [ ]:
requestResult  = sp.artist_top_tracks(artistID, limit=limit, offset=offset)


In [ ]:
DataFrame(retval['items']).T[0]

In [ ]:
retval2 = sp.artist_albums('60d24wfXkVzDSfLS6hyCjZ', limit=50, offset=450)

In [ ]:
retval2['previous']

In [ ]:

results = sp.search(q='weezer', limit=20)
for idx, track in enumerate(results['tracks']['items']):
    print(idx, track['name'])

In [ ]:
result.keys()

In [ ]:
urn = 'spotify:artist:3jOstUTkEu2JkjvRdBA5Gu'

#sp = spotipy.Spotify(client_credentials_manager=SpotifyClientCredentials())

artist = sp.artist(urn)
artist

In [ ]:
result['artists'].keys()

In [ ]:
help(sp.search)

In [ ]:
search_str = 'Radiohead'
result = sp.search(search_str)
pprint.pprint(result)